In [2]:
import os
import sys
from pathlib import Path
import pandas_ta as pta
import numpy as np

# 1. Բոտին հասկացնում ենք, թե որտեղ է գտնվում մեր հիմնական freqtrade պանակը
# Քանի որ մենք user_data/notebooks-ում ենք, երկու քայլ հետ ենք գնում
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

# 2. Ներմուծում ենք Freqtrade-ի տվյալների բեռնման գործիքը
from freqtrade.data.history import load_pair_history
from freqtrade.resolvers import StrategyResolver

print("Միջավայրը հաջողությամբ պատրաստ է:")

Միջավայրը հաջողությամբ պատրաստ է:


In [3]:
from freqtrade.enums import CandleType

# 1. Հասցեն ուղղում ենք դեպի բուն binance պանակը
real_user_data_dir = project_root / 'user_data'

config = {
    'strategy': 'MySupertrand_V5',
    'user_data_dir': real_user_data_dir,
    'trading_mode': 'futures'
}

# Կանչում ենք ստրատեգիան
strategy = StrategyResolver.load_strategy(config)

# 2. Բեռնում ենք ՖՅՈՒՉԵՐՍԱՅԻՆ BTC տվյալները feather ֆորմատով
dataframe = load_pair_history(
    datadir=real_user_data_dir / 'data' / 'binance',
    timeframe=strategy.timeframe,
    pair='BTC/USDT:USDT',
    data_format='feather',            # 🔥 Փոխեցինք feather-ի
    candle_type=CandleType.FUTURES
)

print(f"Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա {len(dataframe)} ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:")

Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա 244927 ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:


In [4]:
# 1. Աշխատեցնում ենք քո ռազմավարության populate_indicators ֆունկցիան
dataframe = strategy.populate_indicators(dataframe, {'pair': 'BTC/USDT:USDT'})

# 2. Էկրանին ենք հանում աղյուսակի վերջին 10 տողերը՝ տեսնելու համար նոր սյունակները
dataframe.tail(10)

,date,open,high,low,close,volume,st_line,st_dir
244917,2026-04-30 09:45:00+00:00,76042.0,76159.6,76035.1,76127.8,329.798,75975.975527,1
244918,2026-04-30 09:50:00+00:00,76127.8,76168.0,76100.0,76111.1,203.983,75975.975527,1
244919,2026-04-30 09:55:00+00:00,76111.2,76111.2,76062.4,76062.7,132.436,75975.975527,1
244920,2026-04-30 10:00:00+00:00,76062.7,76125.5,76054.3,76112.6,182.723,75975.975527,1
244921,2026-04-30 10:05:00+00:00,76112.6,76136.6,76071.4,76110.2,99.186,75975.975527,1
244922,2026-04-30 10:10:00+00:00,76110.2,76114.0,76004.0,76014.4,197.746,75975.975527,1
244923,2026-04-30 10:15:00+00:00,76014.3,76046.1,75984.1,75984.6,262.721,75975.975527,1
244924,2026-04-30 10:20:00+00:00,75984.6,76030.7,75976.3,75976.3,233.632,75975.975527,1
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1


In [5]:
dataframe['atr'] = pta.atr(dataframe['high'], dataframe['low'], dataframe['close'], length=14)
dataframe['prev_high'] = dataframe['high'].shift(1)
dataframe['prev_low'] = dataframe['low'].shift(1)
dataframe.tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low
244922,2026-04-30 10:10:00+00:00,76110.2,76114.0,76004.0,76014.4,197.746,75975.975527,1,88.553575,76136.6,76071.4
244923,2026-04-30 10:15:00+00:00,76014.3,76046.1,75984.1,75984.6,262.721,75975.975527,1,86.656891,76114.0,76004.0
244924,2026-04-30 10:20:00+00:00,75984.6,76030.7,75976.3,75976.3,233.632,75975.975527,1,84.352827,76046.1,75984.1
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1,82.377625,76030.7,75976.3
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1,85.550652,76000.0,75943.3


In [6]:
dataframe['dynamic_sl_long'] = dataframe['prev_low']-dataframe['atr']
dataframe['dynamic_sl_short'] = dataframe['prev_high']+dataframe['atr']
dataframe.tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low,dynamic_sl_long,dynamic_sl_short
244922,2026-04-30 10:10:00+00:00,76110.2,76114.0,76004.0,76014.4,197.746,75975.975527,1,88.553575,76136.6,76071.4,75982.846425,76225.153575
244923,2026-04-30 10:15:00+00:00,76014.3,76046.1,75984.1,75984.6,262.721,75975.975527,1,86.656891,76114.0,76004.0,75917.343109,76200.656891
244924,2026-04-30 10:20:00+00:00,75984.6,76030.7,75976.3,75976.3,233.632,75975.975527,1,84.352827,76046.1,75984.1,75899.747173,76130.452827
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1,82.377625,76030.7,75976.3,75893.922375,76113.077625
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1,85.550652,76000.0,75943.3,75857.749348,76085.550652


In [7]:
dataframe['trend_changed_to_long']= (dataframe['st_dir'].shift(1) == -1) & (dataframe['st_dir'] == 1)
dataframe[dataframe['trend_changed_to_long']==True].tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low,dynamic_sl_long,dynamic_sl_short,trend_changed_to_long
244520,2026-04-29 00:40:00+00:00,76316.3,76391.3,76316.3,76360.0,285.693,76178.277865,1,57.770369,76316.4,76250.0,76192.229631,76374.170369,True
244533,2026-04-29 01:45:00+00:00,76366.8,76575.0,76344.8,76571.8,827.782,76155.622556,1,92.686766,76376.0,76271.6,76178.913234,76468.686766,True
244626,2026-04-29 09:30:00+00:00,77085.6,77200.7,77083.3,77182.5,480.236,76947.228754,1,66.134285,77100.8,77080.3,77014.165715,77166.934285,True
244767,2026-04-29 21:15:00+00:00,75747.4,75843.3,75747.4,75839.4,369.563,75429.614656,1,128.963353,75812.9,75636.9,75507.936647,75941.863353,True
244878,2026-04-30 06:30:00+00:00,75641.7,75818.3,75641.7,75816.9,549.174,75452.091529,1,95.380640,75661.2,75601.2,75505.819360,75756.580640,True


In [17]:
dataframe['entry_price_anchor'] = np.where(dataframe['trend_changed_to_long']==True, dataframe['close'], np.nan)
dataframe['entry_price_anchor'] = dataframe['entry_price_anchor'].ffill()
dataframe.tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low,dynamic_sl_long,dynamic_sl_short,trend_changed_to_long,entry_price_anchor,enter_long,trend_changed_to_short,short_entry_anchor,enter_short
244922,2026-04-30 10:10:00+00:00,76110.2,76114.0,76004.0,76014.4,197.746,75975.975527,1,88.553575,76136.6,76071.4,75982.846425,76225.153575,False,75816.9,0,False,75723.9,0
244923,2026-04-30 10:15:00+00:00,76014.3,76046.1,75984.1,75984.6,262.721,75975.975527,1,86.656891,76114.0,76004.0,75917.343109,76200.656891,False,75816.9,0,False,75723.9,0
244924,2026-04-30 10:20:00+00:00,75984.6,76030.7,75976.3,75976.3,233.632,75975.975527,1,84.352827,76046.1,75984.1,75899.747173,76130.452827,False,75816.9,0,False,75723.9,0
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1,82.377625,76030.7,75976.3,75893.922375,76113.077625,False,75816.9,0,True,75952.8,1
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1,85.550652,76000.0,75943.3,75857.749348,76085.550652,False,75816.9,0,False,75952.8,1


In [9]:
dataframe['enter_long']= np.where((dataframe['st_dir']==1) & (dataframe['close']<=dataframe['entry_price_anchor']), 1, 0)
dataframe[dataframe['enter_long']==True].tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low,dynamic_sl_long,dynamic_sl_short,trend_changed_to_long,entry_price_anchor,enter_long
244884,2026-04-30 07:00:00+00:00,75720.6,75767.9,75696.0,75732.2,246.539,75532.042376,1,82.903187,75745.4,75710.0,75627.096813,75828.303187,False,75816.9,1
244885,2026-04-30 07:05:00+00:00,75732.1,75807.0,75684.6,75787.9,381.191,75532.042376,1,85.724388,75767.9,75696.0,75610.275612,75853.624388,False,75816.9,1
244886,2026-04-30 07:10:00+00:00,75787.8,75810.0,75753.2,75774.9,169.990,75543.152030,1,83.658360,75807.0,75684.6,75600.941640,75890.658360,False,75816.9,1
244887,2026-04-30 07:15:00+00:00,75774.8,75810.0,75751.9,75753.3,226.169,75548.916827,1,81.832763,75810.0,75753.2,75671.367237,75891.832763,False,75816.9,1
244888,2026-04-30 07:20:00+00:00,75753.3,75797.3,75742.5,75797.3,90.025,75548.916827,1,79.901852,75810.0,75751.9,75671.998148,75889.901852,False,75816.9,1


In [16]:
dataframe['trend_changed_to_short'] = np.where((dataframe['st_dir']==-1) & (dataframe['st_dir'].shift(1)==1), True, False )
dataframe['short_entry_anchor'] = np.nan
dataframe.loc[dataframe['trend_changed_to_short']==True, 'short_entry_anchor'] = dataframe['close']
dataframe['short_entry_anchor']=dataframe['short_entry_anchor'].ffill()
dataframe['enter_short'] = 0
dataframe.loc[(dataframe['st_dir']==-1) & (dataframe['close']>= dataframe['short_entry_anchor']), 'enter_short'] =1
dataframe[dataframe['enter_short']==True].tail(5)

,date,open,high,low,close,volume,st_line,st_dir,atr,prev_high,prev_low,dynamic_sl_long,dynamic_sl_short,trend_changed_to_long,entry_price_anchor,enter_long,trend_changed_to_short,short_entry_anchor,enter_short
244848,2026-04-30 04:00:00+00:00,75881.6,75931.7,75787.9,75795.9,447.327,76060.550268,-1,109.274929,75945.9,75880.2,75770.925071,76055.174929,False,75839.4,0,False,75723.9,1
244849,2026-04-30 04:05:00+00:00,75796.0,75810.6,75657.3,75760.7,456.966,76060.550268,-1,112.419577,75931.7,75787.9,75675.480423,76044.119577,False,75839.4,0,False,75723.9,1
244850,2026-04-30 04:10:00+00:00,75760.7,75793.9,75715.4,75748.5,291.285,76060.550268,-1,109.996750,75810.6,75657.3,75547.303250,75920.596750,False,75839.4,0,False,75723.9,1
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1,82.377625,76030.7,75976.3,75893.922375,76113.077625,False,75816.9,0,True,75952.8,1
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1,85.550652,76000.0,75943.3,75857.749348,76085.550652,False,75816.9,0,False,75952.8,1


In [11]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import numpy as np
import plotly.graph_objects as go

# 1. Վերցնում ենք վերջին 500 մոմերը
df_plot = dataframe.iloc[-500:].copy()

# 2. Առանձնացնում ենք Supertrend-ի գծերը
df_plot['st_up'] = np.where(df_plot['st_dir'] == 1, df_plot['st_line'], np.nan)
df_plot['st_down'] = np.where(df_plot['st_dir'] == -1, df_plot['st_line'], np.nan)

# 🔥 Առանձնացնում ենք Մուտքի Սլաքների դիրքերը (որպեսզի միայն 1 եղած տեղերում նկարվեն)
long_signals = df_plot[df_plot['enter_long'] == 1]
short_signals = df_plot[df_plot['enter_short'] == 1]

# 3. Ստեղծում ենք գրաֆիկը
fig = go.Figure()

# 4. Ավելացնում ենք Ճապոնական մոմերը
fig.add_trace(go.Candlestick(
    x=df_plot['date'], open=df_plot['open'], high=df_plot['high'],
    low=df_plot['low'], close=df_plot['close'], name='Գին (BTC)'
))

# 5. Ավելացնում ենք Supertrend-ի գծերը (Կանաչ / Կարմիր)
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['st_up'], mode='lines', name='Supertrend Աճ', line=dict(color='#00FF00', width=2)))
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['st_down'], mode='lines', name='Supertrend Անկում', line=dict(color='#FF0000', width=2)))

# 6. 🔥 Ավելացնում ենք LONG Մուտքի Կանաչ Սլաքները (▲)
fig.add_trace(go.Scatter(
    x=long_signals['date'],
    y=long_signals['low'] * 0.998, # Մոմից մի փոքր ներքև, որ չկպչի
    mode='markers',
    name='LONG Մուտք',
    marker=dict(symbol='triangle-up', size=12, color='#00FF00', line=dict(width=1, color='white'))
))

# 7. 🔥 Ավելացնում ենք SHORT Մուտքի Կարմիր Սլաքները (▼)
fig.add_trace(go.Scatter(
    x=short_signals['date'],
    y=short_signals['high'] * 1.002, # Մոմից մի փոքր վերև
    mode='markers',
    name='SHORT Մուտք',
    marker=dict(symbol='triangle-down', size=12, color='#FF0000', line=dict(width=1, color='white'))
))

# 8. Ձևավորում
fig.update_layout(
    title='BTC/USDT + Ազդանշաններ (Կանաչ ▲ / Կարմիր ▼)',
    yaxis_title='Գին (USDT)', xaxis_title='Ամսաթիվ',
    template='plotly_dark', xaxis_rangeslider_visible=False
)

# fig.show()

# 8. Ցուցադրում ենք գրաֆիկը ավտոմատ բացվող բրաուզերի թաբում
fig.show(renderer="browser")